### Instalando CatBoost

In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.9 MB/s eta 0:00:00


### Instalando bibliiotecas necessárias


In [ ]:
import pandas as pd
import numpy as np
import catboost as cb
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

### Importanto os dados

In [ ]:
df = pd.read_csv("/content/dataset_evasao_sintetico.csv")

/tmp/ipython-input-2386508543.py:1: DtypeWarning: Columns (12,17,18,19,20,21,25,27,29,36) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/dataset_evasao_sintetico.csv")


In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81056 entries, 0 to 81055
Data columns (total 37 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   id_aluno                             81056 non-null  int64  
 1   evasao                               81056 non-null  bool   
 2   sexo                                 81056 non-null  object 
 3   idade                                81056 non-null  int64  
 4   estudante_nis                        81056 non-null  bool   
 5   informou_nome_mae                    81056 non-null  bool   
 6   informou_nome_pai                    81056 non-null  bool   
 7   possui_deficiencia                   81056 non-null  bool   
 8   raca_cor                             81056 non-null  object 
 9   tipo_localizacao_endereco_estudante  81056 non-null  object 
 10  situacao_consolidada_no_ano          81055 non-null  object 
 11  etapa_ensino                

### Pré processamento dos dados

In [ ]:
# Separando as colunas por tipo
numeric_cols = df.select_dtypes(include=['int64', 'float64'])
bool_cols = df.select_dtypes(include=['bool'])
object_cols = df.select_dtypes(include=['object'])

In [ ]:
# Transformando tudo em numeric
bool_to_numeric = bool_cols.astype(int)
object_encoder = pd.get_dummies(object_cols)

In [ ]:
# Juntando tudo
df_final = pd.concat([numeric_cols, bool_to_numeric, object_encoder], axis=1)


In [ ]:
df_final.head() # para simples conferência

,id_aluno,idade,frequencia_escolar,notas_medias,renda_familiar,distancia_da_escola_km,pontuacao_risco,evasao,estudante_nis,informou_nome_mae,...,disponibilidade_material_didatico_Insuficiente,disponibilidade_material_didatico_Parcial,flexibilidade_pedagogica_Alta,flexibilidade_pedagogica_Baixa,flexibilidade_pedagogica_Média,qualidade_pedagogica_percebida_Boa,qualidade_pedagogica_percebida_Regular,qualidade_pedagogica_percebida_Ruim,evasao_confirmada_False,evasao_confirmada_True
0,1,11,92.4,5.9,201.67,0.5,0.216064,0,0,1,...,False,False,True,False,False,True,False,False,True,False
1,2,19,59.3,5.7,334.63,0.7,0.372110,0,1,1,...,False,True,True,False,False,True,False,False,False,True
2,3,22,90.4,8.9,526.12,2.3,0.091583,0,0,1,...,True,False,False,True,False,False,False,True,True,False
3,4,19,70.2,5.6,802.40,4.0,0.328097,0,0,1,...,False,True,True,False,False,False,False,True,False,True
4,5,16,84.6,7.7,1229.73,1.1,0.170717,0,0,1,...,False,False,False,False,True,True,False,False,True,False


### Separando as variáveis

In [ ]:
X = df_final.drop(columns=['id_aluno', 'evasao_confirmada_True', 'evasao_confirmada_False'])
y = df_final['evasao_confirmada_True']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [ ]:
print(f"Total de amostras: {len(X_train) + len(X_test)}")
print(f"Amostras de treino: {len(X_train)} ({len(X_train)/(len(X_train)+len(X_test))*100:.1f}%)")
print(f"Amostras de teste: {len(X_test)} ({len(X_test)/(len(X_train)+len(X_test))*100:.1f}%)")

Total de amostras: 81056
Amostras de treino: 64844 (80.0%)
Amostras de teste: 16212 (20.0%)


### Treinando o modelo

In [ ]:
model = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, verbose=0)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]

### Métricas


In [ ]:
print("Acurácia:", accuracy_score(y_test, y_pred))
print("Precisão:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))
print("F1-Score:", f1_score(y_test, y_pred, average='weighted'))
print("ROC AUC:", roc_auc_score(y_test, y_proba))

Acurácia: 0.694053787318036
Precisão: 0.6579883598839156
Recall: 0.694053787318036
F1-Score: 0.6320005803280153
ROC AUC: 0.6468305947288923


### Análises

In [ ]:
# Importância dos features
feature_importances = model.feature_importances_

importance_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': feature_importances}).sort_values(by='Importance', ascending=False)